<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/saxs_profile_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAXS Profile Generation & Analysis

This tutorial demonstrates how to compute Small-Angle X-ray Scattering (SAXS) profiles directly from atomic coordinates using the Debye formula, and how to analyze the resulting data with Guinier, Kratky, Porod, and pair-distance views.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/elkins-lab/synth-saxs.git biotite matplotlib
else:
    sys.path.append("../../")

In [ ]:
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import numpy as np

## 1. Load a Structure
We will download a PDB file (e.g., Lysozyme, 1AKI) using Biotite.

In [ ]:
import biotite.database.rcsb as rcsb

# Download and load Lysozyme
pdb_file = rcsb.fetch("1AKI", "pdb", ".")
structure = strucio.load_structure(pdb_file)
# Filter out water and hetero atoms
protein = structure[~structure.hetero]

# Extract coordinates and elements
coords = protein.coord
elements = protein.element

print(f"Loaded protein with {len(coords)} atoms.")

## 2. Simulate the SAXS Profile
The Debye scattering formula relates the 3D distances between all pairs of atoms to the 1D scattering intensity $I(q)$:
$$ I(q) = \sum_{i,j} f_i(q) f_j(q) \frac{\sin(q r_{ij})}{q r_{ij}} $$

In [ ]:
from synth_saxs import add_noise, calculate_saxs_profile

# Calculate SAXS profile
q_values, intensities = calculate_saxs_profile(protein, q_min=0.01, q_max=0.5, n_points=100)

# Add synthetic experimental noise
noisy_intensities = add_noise(intensities, noise_level=0.05)

## 3. Visualization
Let's start with a standard $I(q)$ curve and a Kratky plot ($I(q) \cdot q^2$ vs $q$) to assess foldedness.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Standard SAXS profile (Log scale)
ax1.plot(q_values, noisy_intensities, "ko", markersize=4, alpha=0.5, label="Simulated Data")
ax1.plot(q_values, intensities, "r-", linewidth=2, label="Ideal Profile")
ax1.set_yscale("log")
ax1.set_xlabel("q (1/Å)", fontsize=12)
ax1.set_ylabel("Intensity I(q)", fontsize=12)
ax1.set_title("Simulated SAXS Profile", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Kratky Plot
# Folded proteins show a bell-shaped peak, unfolded proteins plateau or increase
kratky_y = intensities * (q_values**2)
ax2.plot(q_values, kratky_y, "b-", linewidth=2)
ax2.set_xlabel("q (1/Å)", fontsize=12)
ax2.set_ylabel("I(q) * q²", fontsize=12)
ax2.set_title("Kratky Plot", fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Complete SAXS Diagnostic Panel
The package visualization helper creates the common SAXS diagnostic views in one call: standard intensity, Kratky, Guinier, and Porod plots.

In [ ]:
from synth_saxs import calculate_radius_of_gyration, plot_saxs_results

rg = calculate_radius_of_gyration(protein)
fig = plot_saxs_results(q_values, intensities, plot_type="all", rg=rg)
plt.show()
print(f"Coordinate Radius of Gyration (Rg): {rg:.2f} Å")

## 5. Guinier Fit: Estimate $R_g$ from Low-q Data
In the Guinier regime, $\ln I(q)$ is approximately linear in $q^2$ with slope $-R_g^2/3$. The fit is only meaningful at low q, typically where $qR_g \lesssim 1.3$.

In [ ]:
guinier_mask = (q_values * rg < 1.3) & (q_values > 0)
q2_low = q_values[guinier_mask] ** 2
ln_i_low = np.log(intensities[guinier_mask])

slope, intercept = np.polyfit(q2_low, ln_i_low, 1)
rg_fit = np.sqrt(max(0, -3 * slope))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(q2_low, ln_i_low, "o", label="Low-q data")
ax.plot(q2_low, slope * q2_low + intercept, "k--", label=f"Fit Rg = {rg_fit:.2f} Å")
ax.set_xlabel("q² (Å⁻²)")
ax.set_ylabel("ln I(q)")
ax.set_title("Guinier Fit")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Coordinate Rg: {rg:.2f} Å")
print(f"Guinier-fit Rg: {rg_fit:.2f} Å")

## 6. Pair Distance Distribution $P(r)$
$P(r)$ is a real-space view of the same structure: it summarizes the distribution of interatomic distances that gives rise to the reciprocal-space SAXS curve.

In [ ]:
from synth_saxs import calculate_p_dist, plot_p_dist

r_values, p_r = calculate_p_dist(protein, bins=80)
fig = plot_p_dist(r_values, p_r)
plt.show()

print(f"Approximate Dmax from P(r): {r_values[p_r > 0].max():.1f} Å")

## 4. Validating Against Experimental Data (Fitting)
Once you have a simulated profile, the ultimate goal is to validate it against real laboratory SAXS data. `synth-saxs` includes a robust fitting engine that uses linear least-squares regression to find the optimal scale ($c$) and offset ($k$) that minimizes the $\chi^2$ error between your simulation and the experiment.

Let's generate some mock "experimental" data by adding noise to our ideal profile, and then use the fitting API.

In [ ]:
import numpy as np

from synth_saxs.fitting import fit_profile

# 1. Create mock experimental data from our intensities
# Scale by 15.0, shift by 0.05, add 2% Gaussian noise
c_true, k_true = 15.0, 0.05
i_exp_ideal = c_true * intensities + k_true
noise = 0.02 * i_exp_ideal
i_exp = np.random.normal(i_exp_ideal, noise)
err_exp = noise

# 2. Save it to a standard .dat file (q, I, err)
exp_data = np.column_stack([q_values, i_exp, err_exp])
np.savetxt("mock_experimental.dat", exp_data, header="q I error")
print("Created 'mock_experimental.dat'")

### Option A: Python API Fitting

In [ ]:
c_fit, k_fit, chi_sq = fit_profile(i_exp, err_exp, intensities)
print(f"Fit Results:\nScale (c) = {c_fit:.3f}\nOffset (k) = {k_fit:.3f}\nchi^2 = {chi_sq:.2f}")

i_fit = c_fit * intensities + k_fit

plt.figure(figsize=(6, 4))
plt.errorbar(
    q_values, i_exp, yerr=err_exp, fmt="ko", markersize=3, alpha=0.5, label="Mock Experiment"
)
plt.semilogy(q_values, i_fit, "r-", linewidth=2, label="Theoretical Fit")
plt.xlabel("q (A^-1)")
plt.ylabel("log I(q)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Option B: CLI Fitting
You can also do all of this directly from the command line using the `--fit` argument!

In [ ]:
!synth-saxs 1AKI.pdb --fit mock_experimental.dat --plot cli_fit.png